# Flow matching figures for `blog.md`

Toy 1D/2D illustrations for the *Flow matching*, *Learning the field* and *Rectified flow*
sections of `blog.md`. Every figure is written to `figs/flow_matching/` as a GIF plus a static
PNG fallback. Nothing here needs a LiDAR checkpoint; the 2D networks train in under a minute.

Conventions follow the blog: $x_0\sim\mathcal N(0,I)$ is noise, $x_1$ is data, time runs from
noise ($t=0$) to data ($t=1$), and the straight-line path is $x_t=(1-t)x_0+t x_1$.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.collections import LineCollection

OUT = Path("../figs/flow_matching")
OUT.mkdir(parents=True, exist_ok=True)

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# palette: grey = noise, red = data, blue/orange = "heading to target A / B"
NOISE, DATA, CA, CB, INK = "#8a9099", "#c8102e", "#1f6fb2", "#e8890c", "#222222"
plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 100, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.labelsize": 11,
})


def save_gif(anim, name, fps=20, dpi=90):
    path = OUT / f"{name}.gif"
    anim.save(path, writer=PillowWriter(fps=fps), dpi=dpi)
    plt.close(anim._fig)
    print(f"wrote {path} ({path.stat().st_size / 1e6:.2f} MB)")


def save_png(fig, name):
    path = OUT / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    print(f"wrote {path}")


def with_holds(n_move, hold_start=10, hold_end=25):
    """Frame index -> progress in [0, 1], pausing at both ends so the GIF loop reads."""
    return np.concatenate([np.zeros(hold_start), np.linspace(0, 1, n_move), np.ones(hold_end)])

## 1. The sliding jam (section "A worked jam")

$\rho(x,t)=100\,\mathcal N(x-ct;0,1)$ with $c=1$ km/min. Three observers sit 1 km behind the jam,
at its peak, and 1 km ahead. Each shows its current density and the sign of
$\partial_t\rho=-c\,\partial_x\rho$.

In [2]:
c = 1.0
xs = np.linspace(-4, 5, 600)
rho = lambda x, t: 100 / np.sqrt(2 * np.pi) * np.exp(-0.5 * (x - c * t) ** 2)
drho_dt = lambda x, t: (x - c * t) * rho(x, t) * c  # -c * d/dx rho
observers = [(-1.0, "behind"), (0.0, "at peak"), (1.0, "ahead")]
obs_col = [CA, INK, CB]


def draw_observer(ax, xo, label, col, t, arrows=True):
    r, d = rho(xo, t), drho_dt(xo, t)
    ax.axvline(xo, color=col, ls=":", lw=1)
    ax.plot([xo], [r], "o", color=col, ms=7, zorder=5)
    arrow = "" if abs(d) < 0.5 else ("↑" if d > 0 else "↓")
    ax.text(xo, 47 + 6 * (xo == 0), f"{label}\nρ={r:4.1f} {arrow}", color=col, ha="center", va="bottom", fontsize=9)


fig, ax = plt.subplots(figsize=(8, 3.8))
ts = with_holds(40)


def frame(i):
    t = ts[i]
    ax.clear()
    ax.fill_between(xs, rho(xs, 0), color=NOISE, alpha=0.15, lw=0)
    ax.plot(xs, rho(xs, 0), color=NOISE, ls="--", lw=1, label=r"$\rho_0(x)$")
    ax.plot(xs, rho(xs, t), color=DATA, lw=2.2, label=r"$\rho(x,t)$")
    for (xo, lab), col in zip(observers, obs_col):
        draw_observer(ax, xo, lab, col, t)
    ax.set_xlim(-4, 5); ax.set_ylim(0, 64)
    ax.set_xlabel("position x (km)"); ax.set_ylabel("cars / km")
    ax.set_title(f"constant speed c = 1 km/min:   t = {t:.2f} min", loc="left")
    ax.legend(loc="upper right", frameon=False)


anim = FuncAnimation(fig, frame, frames=len(ts))
save_gif(anim, "traffic_jam")

# static: t = 0 vs t = 1 with the instantaneous trends at t = 0 (the table in the blog)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(xs, rho(xs, 0), color=NOISE, lw=2, label=r"$t=0$")
ax.plot(xs, rho(xs, 1), color=DATA, lw=2, label=r"$t=1$ min")
for (xo, lab), col in zip(observers, obs_col):
    draw_observer(ax, xo, lab, col, 0)
    ax.plot([xo], [rho(xo, 1)], "s", color=col, ms=6, mfc="white", zorder=5)
ax.annotate("", xy=(1.0, 42), xytext=(0.0, 42), arrowprops=dict(arrowstyle="->", color=DATA))
ax.set_xlim(-4, 5); ax.set_ylim(0, 64)
ax.set_xlabel("position x (km)"); ax.set_ylabel("cars / km")
ax.legend(loc="upper right", frameon=False)
save_png(fig, "traffic_jam"); plt.close(fig)

wrote ../figs/flow_matching/traffic_jam.gif (0.25 MB)
wrote ../figs/flow_matching/traffic_jam.png


## 2. One scan: the conditional path (Eqs. 7–9)

Noise samples drive in straight lines at constant speed to a single data point $x_1$.
The cloud is $p_t(x\mid x_1)=\mathcal N(t x_1,(1-t)^2 I)$: the mean slides, the spread shrinks.

In [3]:
x1 = np.array([2.6, 1.6])
x0 = rng.standard_normal((160, 2))
theta = np.linspace(0, 2 * np.pi, 100)
ts = with_holds(50, hold_start=8, hold_end=20)

fig, ax = plt.subplots(figsize=(5.2, 4.6))


def frame(i):
    t = ts[i]
    ax.clear()
    for a in x0:
        ax.plot([a[0], x1[0]], [a[1], x1[1]], color=NOISE, lw=0.6, alpha=0.45)
    xt = (1 - t) * x0 + t * x1
    for k, a in zip((1, 2), (0.9, 0.5)):  # 1- and 2-sigma circles of p_t(x | x1)
        r = k * (1 - t)
        ax.plot(t * x1[0] + r * np.cos(theta), t * x1[1] + r * np.sin(theta), color=CA, lw=1.8, alpha=a)
    ax.scatter(xt[:, 0], xt[:, 1], s=10, color=CA, zorder=3)
    ax.plot(*x1, "*", color=DATA, ms=16, zorder=4)
    ax.text(x1[0] + 0.15, x1[1] + 0.2, r"$x_1$", color=DATA, fontsize=13)
    ax.set_xlim(-3.2, 3.8); ax.set_ylim(-3.2, 3.2); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(rf"$p_t(x\mid x_1)=\mathcal{{N}}(t\,x_1,(1-t)^2I)$,   t = {t:.2f}", loc="left", fontsize=11)


anim = FuncAnimation(fig, frame, frames=len(ts))
save_gif(anim, "conditional_path")

wrote ../figs/flow_matching/conditional_path.gif (0.34 MB)


## Exact marginal field for a finite dataset (Eq. 14)

$u_t(x)=\sum_i w_i(t,x)\,\frac{x_1^{(i)}-x}{1-t}$ with $w_i\propto\mathcal N(x;t x_1^{(i)},(1-t)^2I)$.
Integrated with Euler on the grid $t_k=k/K$; the last step has $\Delta t/(1-t)=1$ and lands exactly on
the posterior mean $\mathbb E[x_1\mid x_t]$, so the singularity at $t=1$ is harmless.

In [4]:
def exact_u(t, x, data):
    """x: (N, d), data: (M, d) -> (N, d). Posterior-weighted average of conditional velocities."""
    diff = x[:, None, :] - t * data[None, :, :]
    logw = -0.5 * (diff ** 2).sum(-1) / (1 - t) ** 2
    w = np.exp(logw - logw.max(1, keepdims=True))
    w /= w.sum(1, keepdims=True)
    return (w @ data - x) / (1 - t)


def euler(v, x0, K):
    xs_ = [x0]
    for k in range(K):
        t = k / K
        xs_.append(xs_[-1] + v(t, xs_[-1]) / K)
    return np.stack(xs_)  # (K+1, N, d)

## 3. Two-point example: why the trajectories bend

Data $\{-1,+1\}$. Left: the conditional straight lines for random noise/data pairs; they cross.
Right: trajectories of the averaged field $u_t$. They hesitate near $x=0$, then commit and bend;
they never cross. Background is $u_t(x)$ (blue = moving down, red = moving up).

In [5]:
data1 = np.array([[-1.0], [1.0]])
n = 26
x0_1d = np.sort(rng.standard_normal(n))[:, None]
target = rng.choice(data1[:, 0], size=n)[:, None]  # independent (random) coupling
K = 200
tgrid = np.arange(K + 1) / K
lines_cond = (1 - tgrid)[:, None, None] * x0_1d[None] + tgrid[:, None, None] * target[None]
lines_marg = euler(lambda t, x: exact_u(t, x, data1), x0_1d, K)
col_cond = np.where(target[:, 0] > 0, CB, CA)
col_marg = np.where(lines_marg[-1, :, 0] > 0, CB, CA)

# heatmap of u_t(x), compressed so the 1/(1-t) blow-up does not wash it out
TT, XX = np.meshgrid(np.linspace(0, 0.995, 200), np.linspace(-2.6, 2.6, 200))
U = np.stack([exact_u(t, XX[:, j:j + 1], data1)[:, 0] for j, t in enumerate(TT[0])], 1)
U = np.tanh(U / 2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
idx = np.round(with_holds(60, hold_start=6, hold_end=25) * K).astype(int)


def setup(ax, title):
    ax.set_xlim(0, 1); ax.set_ylim(-2.6, 2.6)
    ax.set_xlabel("t"); ax.set_title(title, loc="left")
    ax.plot([1, 1], [-1, 1], "o", color=DATA, ms=9, zorder=5, clip_on=False)


axes[1].imshow(U, extent=(0, 1, -2.6, 2.6), origin="lower", aspect="auto", cmap="RdBu_r", vmin=-1.3, vmax=1.3, alpha=0.35)
setup(axes[0], "conditional paths (straight, cross)")
setup(axes[1], r"averaged field $u_t$ (bend, never cross)")
axes[0].set_ylabel("x")
lc0 = LineCollection([], colors=col_cond, lw=1.3); axes[0].add_collection(lc0)
lc1 = LineCollection([], colors=col_marg, lw=1.3); axes[1].add_collection(lc1)
dots0 = axes[0].scatter(np.zeros(n), x0_1d[:, 0], s=14, c=col_cond, zorder=4)
dots1 = axes[1].scatter(np.zeros(n), x0_1d[:, 0], s=14, c=col_marg, zorder=4)
fig.tight_layout()


def frame(i):
    k = max(idx[i], 1)
    lc0.set_segments([np.c_[tgrid[:k + 1], lines_cond[:k + 1, j, 0]] for j in range(n)])
    lc1.set_segments([np.c_[tgrid[:k + 1], lines_marg[:k + 1, j, 0]] for j in range(n)])
    dots0.set_offsets(np.c_[np.full(n, tgrid[k]), lines_cond[k, :, 0]])
    dots1.set_offsets(np.c_[np.full(n, tgrid[k]), lines_marg[k, :, 0]])
    return lc0, lc1, dots0, dots1


anim = FuncAnimation(fig, frame, frames=len(idx))
save_gif(anim, "two_point")
frame(len(idx) - 1)
save_png(fig, "two_point"); plt.close(fig)

wrote ../figs/flow_matching/two_point.gif (0.33 MB)
wrote ../figs/flow_matching/two_point.png


## 4. The marginal field in 2D (Eqs. 12–14)

Eight "scans" on a ring. Arrows show the expected remaining displacement
$(1-t)\,u_t(x)=\mathbb E[x_1\mid x_t=x]-x$. Early on every point is pulled toward the centre of
the dataset; as $t\to1$ the posterior sharpens, the field splits into basins, and every particle is
pulled exactly onto one training point.

In [6]:
M = 8
ang = 2 * np.pi * np.arange(M) / M + np.pi / 8
ring = 2.2 * np.c_[np.cos(ang), np.sin(ang)]
x0_2d = rng.standard_normal((400, 2))
K = 100
traj_exact = euler(lambda t, x: exact_u(t, x, ring), x0_2d, K)

g = np.linspace(-3.2, 3.2, 21)
GX, GY = np.meshgrid(g, g)
grid = np.c_[GX.ravel(), GY.ravel()]
idx = np.round(with_holds(60, hold_start=8, hold_end=25) * K).astype(int)
idx = np.minimum(idx, K)

fig, ax = plt.subplots(figsize=(5.2, 5.2))
quiv = ax.quiver(GX, GY, np.zeros_like(GX), np.zeros_like(GY), color=NOISE, angles="xy", scale_units="xy", scale=1, width=0.004, alpha=0.8)
pts = ax.scatter(*x0_2d.T, s=6, color=CA, alpha=0.7, zorder=3)
ax.scatter(*ring.T, marker="*", s=220, facecolors="none", edgecolors=DATA, lw=1.4, zorder=4)
ax.set_xlim(-3.4, 3.4); ax.set_ylim(-3.4, 3.4); ax.set_aspect("equal"); ax.axis("off")
title = ax.set_title(r"$\mathbb{E}[x_1\mid x_t{=}x]-x$,   t = 0.00", loc="left")
fig.tight_layout()


def frame(i):
    k = idx[i]
    t = min(k / K, 0.995)
    d = exact_u(t, grid, ring) * (1 - t)  # = E[x1 | x] - x
    d *= 0.35  # visual scale
    quiv.set_UVC(d[:, 0].reshape(GX.shape), d[:, 1].reshape(GX.shape))
    pts.set_offsets(traj_exact[k])
    title.set_text(rf"$\mathbb{{E}}[x_1\mid x_t{{=}}x]-x$,   t = {k / K:.2f}")
    return quiv, pts, title


anim = FuncAnimation(fig, frame, frames=len(idx))
save_gif(anim, "marginal_field")

wrote ../figs/flow_matching/marginal_field.gif (1.06 MB)


## 5–6. Exact field vs. a learned field, and the loss floor

Train a small MLP $v_\theta(t,x)$ with the conditional flow matching loss (Eq. 18) on the same 8
points. While training we also track
* $\mathcal L_{\mathrm{FM}}=\mathbb E\|v_\theta-u_t\|^2$ against the exact field, and
* the floor $C=\mathbb E\,\|(x_1-x_0)-u_t(x_t)\|^2$ (Eq. 17),

so we can check $\mathcal L_{\mathrm{CFM}}=\mathcal L_{\mathrm{FM}}+C$ numerically.

The network is deliberately small. A big one trained long enough memorises the 8 points just like
the exact field; a small one cannot reproduce the sharp switching near $t=1$, so most samples land
on the ring *between* training points.

In [7]:
class MLP(nn.Module):
    def __init__(self, d=2, h=128, depth=3):
        super().__init__()
        layers, w = [], d + 1
        for _ in range(depth):
            layers += [nn.Linear(w, h), nn.SiLU()]
            w = h
        self.net = nn.Sequential(*layers, nn.Linear(h, d))

    def forward(self, t, x):
        return self.net(torch.cat([x, t[:, None]], 1))


def cfm_batch(sample_x1, B):
    x1 = sample_x1(B)
    x0 = torch.randn_like(x1)
    t = torch.rand(B, device=x1.device)
    xt = (1 - t[:, None]) * x0 + t[:, None] * x1
    return x0, x1, t, xt


def train_cfm(model, sample_x1, steps, B=512, lr=2e-3, log=None, log_every=50, sample_x0x1=None):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    losses = []
    for s in range(steps):
        if sample_x0x1 is None:
            x0, x1, t, xt = cfm_batch(sample_x1, B)
        else:  # reflow: fixed (x0, x1) couplings
            x0, x1 = sample_x0x1(B)
            t = torch.rand(B, device=x1.device)
            xt = (1 - t[:, None]) * x0 + t[:, None] * x1
        loss = ((model(t, xt) - (x1 - x0)) ** 2).sum(1).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        losses.append(loss.item())
        if log is not None and s % log_every == 0:
            log(s, model)
    return np.array(losses)


@torch.no_grad()
def sample_ode(model, x0, K):
    xs_ = [x0]
    for k in range(K):
        t = torch.full((x0.shape[0],), k / K, device=x0.device)
        xs_.append(xs_[-1] + model(t, xs_[-1]) / K)
    return torch.stack(xs_).cpu().numpy()


ring_t = torch.tensor(ring, dtype=torch.float32, device=DEVICE)
sample_ring = lambda B: ring_t[torch.randint(0, M, (B,), device=DEVICE)]

# fixed evaluation batch with the exact field as ground truth
x0e, x1e, te, xte = cfm_batch(sample_ring, 20000)
te = te.clamp(max=1 - 1e-3)
xte = (1 - te[:, None]) * x0e + te[:, None] * x1e
u_true = torch.tensor(
    np.concatenate([exact_u(t, x[None], ring) for t, x in zip(te.cpu().numpy(), xte.cpu().numpy().astype(np.float64))]),
    dtype=torch.float32, device=DEVICE)
C_floor = ((x1e - x0e) - u_true).pow(2).sum(1).mean().item()

fm_log = []


@torch.no_grad()
def log_fm(s, model):
    v = model(te, xte)
    fm_log.append((s, (v - u_true).pow(2).sum(1).mean().item(), (v - (x1e - x0e)).pow(2).sum(1).mean().item()))


torch.manual_seed(SEED)
net_ring = MLP(h=32, depth=3).to(DEVICE)  # small on purpose: too smooth to memorise 8 points
losses_ring = train_cfm(net_ring, sample_ring, steps=3000, log=log_fm)
fm_log = np.array(fm_log)
print(f"C = {C_floor:.3f};  final eval L_CFM = {fm_log[-1, 2]:.3f},  L_FM + C = {fm_log[-1, 1] + C_floor:.3f}")

C = 3.584;  final eval L_CFM = 4.134,  L_FM + C = 4.122


In [8]:
K = 100
traj_learned = sample_ode(net_ring, torch.tensor(x0_2d, dtype=torch.float32, device=DEVICE), K)
idx = np.minimum(np.round(with_holds(60, hold_start=8, hold_end=30) * K).astype(int), K)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.7))
scs, titles = [], []
names = ["exact field (Eq. 14)", r"learned $v_\theta$"]
for ax, name in zip(axes, names):
    ax.add_patch(plt.Circle((0, 0), 2.2, fill=False, ls="--", color=NOISE, lw=0.8))
    scs.append(ax.scatter(*x0_2d.T, s=6, color=CA, alpha=0.6, zorder=3))
    ax.scatter(*ring.T, marker="*", s=220, facecolors="none", edgecolors=DATA, lw=1.4, zorder=4)
    ax.set_xlim(-3.4, 3.4); ax.set_ylim(-3.4, 3.4); ax.set_aspect("equal"); ax.axis("off")
    titles.append(ax.set_title(f"{name},   t = 0.00", loc="left"))
fig.tight_layout()


def frame(i):
    k = idx[i]
    scs[0].set_offsets(traj_exact[k]); scs[1].set_offsets(traj_learned[k])
    for ttl, name in zip(titles, names):
        ttl.set_text(f"{name},   t = {k / K:.2f}")
    return (*scs, *titles)


anim = FuncAnimation(fig, frame, frames=len(idx))
save_gif(anim, "exact_vs_learned")
frame(len(idx) - 1)
for ttl, name in zip(titles, names):
    ttl.set_text(name)
save_png(fig, "exact_vs_learned"); plt.close(fig)

wrote ../figs/flow_matching/exact_vs_learned.gif (0.72 MB)
wrote ../figs/flow_matching/exact_vs_learned.png


In [9]:
ema = np.copy(losses_ring)
for i in range(1, len(ema)):
    ema[i] = 0.98 * ema[i - 1] + 0.02 * losses_ring[i]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(losses_ring, color=NOISE, lw=0.5, alpha=0.5)
ax.plot(ema, color=CA, lw=2, label=r"$\mathcal{L}_{\mathrm{CFM}}$ (training loss, smoothed)")
ax.plot(fm_log[:, 0], fm_log[:, 1], color=CB, lw=2, label=r"$\mathcal{L}_{\mathrm{FM}}$ vs. exact field")
ax.axhline(C_floor, color=DATA, ls="--", lw=1.5, label=rf"floor $C$ = {C_floor:.2f}")
ax.set_yscale("log"); ax.set_ylim(min(fm_log[:, 1].min(), C_floor) * 0.5, ema.max() * 1.5)
ax.set_xlabel("training step"); ax.set_ylabel("squared error")
ax.legend(frameon=False)
save_png(fig, "cfm_loss_floor"); plt.close(fig)

wrote ../figs/flow_matching/cfm_loss_floor.png


## 7. Rectified flow: reflow straightens the trajectories

Gaussian $\to$ two moons. 1-RF is trained on random noise/data pairs. We then run its ODE to get
coupled pairs $(x_0,\hat x_1)$ and train 2-RF on those (initialised from 1-RF, as in Liu et al.).
The GIF draws trajectories from the same noise for both; the PNG shows samples with 1, 2, 4 and
100 Euler steps.

In [10]:
def moons(B, noise=0.06):
    s = torch.rand(B, device=DEVICE) * np.pi
    up = torch.rand(B, device=DEVICE) < 0.5
    x = torch.where(up, torch.cos(s), 1 - torch.cos(s))
    y = torch.where(up, torch.sin(s), 0.5 - torch.sin(s))
    pts_ = torch.stack([x - 0.5, y - 0.25], 1) * 2.0
    return pts_ + noise * torch.randn_like(pts_)


torch.manual_seed(SEED)
rf1 = MLP(h=256, depth=4).to(DEVICE)
train_cfm(rf1, moons, steps=8000, B=1024, lr=1e-3)

# reflow: generate coupled pairs with the 1-RF ODE, then fine-tune a copy on them
with torch.no_grad():
    z0 = torch.randn(100_000, 2, device=DEVICE)
    z1 = torch.tensor(sample_ode(rf1, z0, 100)[-1], device=DEVICE)
rf2 = MLP(h=256, depth=4).to(DEVICE)
rf2.load_state_dict(rf1.state_dict())


def sample_pairs(B):
    i = torch.randint(0, z0.shape[0], (B,), device=DEVICE)
    return z0[i], z1[i]


train_cfm(rf2, None, steps=8000, B=1024, lr=1e-3, sample_x0x1=sample_pairs)

array([1.01295686e+00, 7.11856127e-01, 4.97953266e-01, ...,
       2.01522926e-04, 1.06662657e-04, 1.96931011e-04])

In [11]:
torch.manual_seed(1)
x0_show = torch.randn(220, 2, device=DEVICE)
K = 100
tr1 = sample_ode(rf1, x0_show, K)
tr2 = sample_ode(rf2, x0_show, K)
data_bg = moons(3000).cpu().numpy()


def straightness(tr):
    """Mean squared deviation of the velocity from the straight-line displacement (0 = straight)."""
    v = np.diff(tr, axis=0) * (tr.shape[0] - 1)
    return float(((v - (tr[-1] - tr[0])) ** 2).sum(-1).mean())


S1, S2 = straightness(tr1), straightness(tr2)
print(f"straightness  1-RF: {S1:.4f}   2-RF: {S2:.4f}")

idx = np.minimum(np.round(with_holds(60, hold_start=6, hold_end=30) * K).astype(int), K)
cols = np.where(tr1[-1, :, 0] < 0, CA, CB)  # colour by which side the 1-RF sample lands on
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.8))
art = []
for ax, tr, name, S in zip(axes, [tr1, tr2], ["1-RF (random pairs)", "2-RF (after reflow)"], [S1, S2]):
    ax.scatter(*data_bg.T, s=2, color=DATA, alpha=0.12)
    lc = LineCollection([], colors=cols, lw=0.7, alpha=0.6); ax.add_collection(lc)
    sc = ax.scatter(*tr[0].T, s=7, c=cols, zorder=3)
    ax.set_xlim(-3.3, 3.3); ax.set_ylim(-3.3, 3.3); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"{name}\nstraightness S = {S:.3f}", loc="left")
    art.append((lc, sc, tr))
fig.tight_layout()


def frame(i):
    k = max(idx[i], 1)
    for lc, sc, tr in art:
        lc.set_segments([tr[:k + 1, j] for j in range(tr.shape[1])])
        sc.set_offsets(tr[k])
    return [a for lc, sc, _ in art for a in (lc, sc)]


anim = FuncAnimation(fig, frame, frames=len(idx))
save_gif(anim, "reflow")

# static: few-step Euler sampling quality
steps_list = [1, 2, 4, 100]
x0_many = torch.randn(3000, 2, device=DEVICE)
fig, axes = plt.subplots(2, len(steps_list), figsize=(3 * len(steps_list), 6.2))
for r, (net, name) in enumerate([(rf1, "1-RF"), (rf2, "2-RF")]):
    for cidx, K_ in enumerate(steps_list):
        ax = axes[r, cidx]
        s = sample_ode(net, x0_many, K_)[-1]
        ax.scatter(*data_bg.T, s=2, color=DATA, alpha=0.12)
        ax.scatter(*s.T, s=2, color=CA, alpha=0.5)
        ax.set_xlim(-3, 3); ax.set_ylim(-2.4, 2.4); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_visible(False)
        if r == 0:
            ax.set_title(f"{K_} Euler step{'s' if K_ > 1 else ''}")
        if cidx == 0:
            ax.set_ylabel(name, fontsize=12)
fig.tight_layout()
save_png(fig, "reflow"); plt.close(fig)

straightness  1-RF: 1.2664   2-RF: 0.0001


wrote ../figs/flow_matching/reflow.gif (1.23 MB)
wrote ../figs/flow_matching/reflow.png
